# Import Modules

In [1]:
import pandas as pd
import numpy as np

import os
from datetime import datetime
import pandas as pd
from datetime import datetime as dt

# Enable import of other modules
import sys
ROOT = r"C:\Users\g.varvounis\Documents\RiC"
if ROOT not in sys.path:
    sys.path.append(ROOT)

# Functions

In [2]:
def create_path(directory: str = '', filename: str = ''):
    """Creates the full path of files."""
    full_path = os.path.join(directory, filename)

    return full_path

In [3]:
def fetch_api_data(api: str, risk_factor: str, mode: str = 'DA', date_format: str = '%Y-%m-%d %H:%M:%S', 
                   resample_frequency: str = 'D', peak_hour_1: str = "08:00", peak_hour_2: str = "19:00", output_dir: str = ''):
        """Fetches data from Perseo."""
        # Fetch Data from API
        df = pd.read_csv(api)
        df_hour = None
        # Convert Date to Datetime data type & set as index of DataFrame
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', drop=True, inplace=True)

        # Keep aside values for the PK MtD / YtD Average Calculation
        df_pk = df.dropna().copy()

            # Keep hours of inerest
        df_pk = df_pk.between_time(peak_hour_1, peak_hour_2)
            # keep weekdays only
        df_pk = df_pk[df_pk.index.dayofweek < 5]
        df_pk.index = pd.to_datetime(df_pk.index)
        
        df_pk_ytd = df_pk.resample("YE").mean()
        df_pk_mtd = df_pk.resample("ME").mean()
        #############################################################

        # Rename index
        df.index.name = None
        df.index = pd.to_datetime(df.index)
        df_hour=df.copy()
        if 'DA' in mode:
            df_pk_ytd.to_excel(create_path(output_dir, risk_factor + " - Peak YtD.xlsx"))
            df_pk_mtd.to_excel(create_path(output_dir, risk_factor + " - Peak MtD.xlsx"))
            
            df.to_excel(create_path(output_dir, risk_factor + " - Initial Frequency.xlsx"))
            df_pk.to_excel(create_path(output_dir, risk_factor + " - Peak Only.xlsx"))
            df = df.resample(resample_frequency).mean()

        # Rename column of DataFrame
        df.columns = [risk_factor]
        df_mtd = df.resample("ME").mean()
        df_ytd = df.resample("YE").mean()
        #df_wtd = df[df.index.dayofweek<5].resample("W-FRI").mean()
        df_wtd = df.resample("W-SUN").mean()

        return df, df_mtd, df_ytd, df_hour, df_wtd

# Parameters

## Dir

In [4]:
output_dir = r"C:\Users\g.varvounis\Documents\RiC\_data\SPOT"

## Dates

In [5]:
start_date="01-01-2020"
end_date= "10-11-2026"

ytd_tables_start= "2026-01-01"
mtd_tables_start= "2026-11-01"

## API Keys

In [6]:
spot_tickers={
        "GR POWER": "http://10.7.32.34/cdm/Quotations/GetRData/91535c0a-31eb-4906-b895-2cd0fdcf36eb?startDate=",
        "HU POWER": "http://10.7.32.34/cdm/Quotations/GetRData/a48f70d6-a65e-4dcb-911f-23e8070a2786?startDate=",
        "DE POWER": "http://10.7.32.34/cdm/Quotations/GetRData/2ce30a07-6941-4abb-87b4-4a581df45ba2?startDate=",
        "FR POWER": "http://10.7.32.34/cdm/Quotations/GetRData/646aac87-16ef-4479-9544-01c6126e720e?startDate=",
        "BG POWER": "http://10.7.32.34/cdm/Quotations/GetRData/cca6339e-9b67-48b9-b36e-b4ffbb1cad56?startDate=",
        "RO POWER": "http://10.7.32.34/cdm/Quotations/GetRData/596054ed-3ec5-4953-b0d0-c91426c357c6?startDate="}

In [7]:
ttf_tickers={
        "ICIS_TTF_FUTURE_DA+": "http://10.7.32.34/cdm/Quotations/GetRData/225691ab-9366-469c-8739-139a38b91a23?startDate=",
        "ICIS_TTF_FUTURE_M1+": "http://10.7.32.34/cdm/Quotations/GetRData/0aa94acb-dc5c-4f6a-87f6-842b886583b5?startDate=",
        "ICIS_TTF_FUTURE_M2+": "http://10.7.32.34/cdm/Quotations/GetRData/94a61823-8fba-472c-b856-4cc7e7794e7e?startDate=",
        "ICIS_TTF_FUTURE_M3+": "http://10.7.32.34/cdm/Quotations/GetRData/d7a4cf29-7b48-4d4a-854f-6091ca3ad1cb?startDate=",
        "ICIS_TTF_FUTURE_Q1+": "http://10.7.32.34/cdm/Quotations/GetRData/9a3e498c-06d1-4562-bd84-62715623efbe?startDate=",
        "ICIS_TTF_FUTURE_Q2+": "http://10.7.32.34/cdm/Quotations/GetRData/7604674f-e457-441d-884e-893d6d1ddd57?startDate=",
        "ICIS_TTF_FUTURE_Q3+": "http://10.7.32.34/cdm/Quotations/GetRData/03997392-51aa-4f95-a4ee-3aaa0dfb061f?startDate=",
        "ICIS_TTF_FUTURE_Q4+": "http://10.7.32.34/cdm/Quotations/GetRData/29390e9e-a644-41ee-b821-a825a889a09b?startDate=",
        "ICIS_TTF_FUTURE_Q5+": "http://10.7.32.34/cdm/Quotations/GetRData/2c6a9344-55a6-44be-8b24-5d95521d6f3d?startDate=",
        "ICIS_TTF_FUTURE_Y1+": "http://10.7.32.34/cdm/Quotations/GetRData/e65aef94-e121-4715-87ce-e48bdbf37f00?startDate=",
        "ICIS_TTF_FUTURE_Y2+": "http://10.7.32.34/cdm/Quotations/GetRData/79c6cb1c-6888-42db-aa06-645433370532?startDate=",}

In [8]:
api_mid= "&endDate="
api_end="&dates=true&simulation=0"

# Executable Code

In [9]:
mtd_values = []
ytd_values = []
wtd_values=[]
spot_values = []
ticker_labels = []

for idx, ticker in enumerate(spot_tickers.keys()):
    api = spot_tickers[ticker]
    api = api + start_date + api_mid + end_date + api_end

    df_temp, df_temp_mtd, df_temp_ytd, df_hour, df_temp_wtd = fetch_api_data(api=api, risk_factor=ticker, output_dir=output_dir)
       
    if idx==0:
            
        df_mtd_matrix = df_temp_mtd.copy()
        df_ytd_matrix = df_temp_ytd.copy()
        df_wtd_matrix = df_temp_wtd.copy()
        df_hours_all_countries = df_hour.copy()
        df_hours_all_countries["COUNTRY"] = np.nan
        df_hours_all_countries["COUNTRY"] =  df_hours_all_countries["COUNTRY"].fillna(ticker)
    else:
        df_mtd_matrix = pd.concat([df_mtd_matrix, df_temp_mtd], axis=1)
        df_ytd_matrix = pd.concat([df_ytd_matrix, df_temp_ytd], axis=1)
        df_wtd_matrix = pd.concat([df_wtd_matrix, df_temp_wtd], axis=1)
           
        df_hour_temp = df_hour.copy()
        df_hour_temp["COUNTRY"] = np.nan
        df_hours_all_countries = pd.concat([df_hours_all_countries, df_hour_temp], axis=0)
        df_hours_all_countries["COUNTRY"] =  df_hours_all_countries["COUNTRY"].fillna(ticker)

        
    mtd_values.append(df_temp_mtd.iloc[-1,0])
    ytd_values.append(df_temp_ytd.iloc[-1,0])
    wtd_values.append(df_temp_wtd.iloc[-1,0])
    spot_values.append(df_temp.iloc[-1,0])
    ticker_labels.append(ticker)

    df_temp.to_excel(create_path(output_dir, ticker + " AVG.xlsx"))
    df_temp_mtd.to_excel(create_path(output_dir, ticker + " MTD.xlsx"))
    df_temp_ytd.to_excel(create_path(output_dir, ticker + " YTD.xlsx"))
    df_temp_wtd.to_excel(create_path(output_dir, ticker + " WTD.xlsx"))

df_mtd_matrix.columns = list(spot_tickers.keys())
    
df_ytd_matrix.columns = list(spot_tickers.keys())

df_wtd_matrix.columns = list(spot_tickers.keys())

    
df_summary_values_mtd = pd.DataFrame({"Country": ticker_labels, "MtD": mtd_values})
df_summary_values_ytd = pd.DataFrame({"Country": ticker_labels, "YtD": ytd_values})
df_summary_values_wtd = pd.DataFrame({"Country": ticker_labels, "YtD": wtd_values})
df_summary_values_da = pd.DataFrame({"Country": ticker_labels, "DA": spot_values})

df_summary_values_mtd.to_excel(create_path(output_dir, "MtD Summary Table.xlsx"))
df_summary_values_ytd.to_excel(create_path(output_dir, "YtD Summary Table.xlsx"))
df_summary_values_wtd.to_excel(create_path(output_dir, "WtD Summary Table.xlsx"))
df_summary_values_da.to_excel(create_path(output_dir, "DA Summary Table.xlsx"))

df_mtd_matrix.to_excel(create_path(output_dir, "MtD Matrix.xlsx"))
df_wtd_matrix.to_excel(create_path(output_dir, "WtD Matrix.xlsx"))
df_ytd_matrix.to_excel(create_path(output_dir, "YtD Matrix.xlsx"))

df_hours_all_countries.to_excel(create_path(output_dir, "DA POWER HOUR.xlsx"))

In [10]:
dict_ttf = {}

for idx, ttf_ticker in enumerate(ttf_tickers.keys()):
    api = ttf_tickers[ttf_ticker]
    api = api + start_date + api_mid + end_date + api_end

    df_temp = pd.read_csv(api)
    df_temp.set_index('date', drop=True, inplace=True)
    df_temp.index.name = None
       
    df_temp.to_excel(create_path(output_dir, str(ttf_ticker) + ".xlsx"))